## Microproyecto 2

Instalar las librerias necesarias

In [ ]:
# Instalación de librerías
%pip install nltk
%pip install openpyxl

# Librerías generales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# NLTK
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


# Procesamiento de texto
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

# Excel
import openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jsroj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jsroj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jsroj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Llamar los datos / Revisar si hay duplicados o faltantes

In [ ]:
data = pd.read_excel('datos_microproy_2.xlsx')

In [ ]:
data

,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5
...,...,...
9651,Esto implica que el tiempo de las mujeres en e...,5
9652,"Sin embargo, estas fallas del mercado implican...",3
9653,El hecho de hacerlo y cómo hacerlo dependerá e...,9
9654,"Esto se destacó en el primer estudio de caso, ...",6


In [ ]:
## Duplicados o faltantes 
print(data.isna().sum())
print(data.duplicated().sum())

textos    0
ODS       0
dtype: int64
0


Dividir en variables explicativas X y objetivo y

In [ ]:
X= data['textos']
y= data['ODS']

Dividir el set de datos en train y test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

 1. Preparación de los textos utilizando el esquema de bolsa de palabras (BOW)l con una pesado TF-IDF. Para este paso construir un pipeline que integre las transformaciones que se consideren adecuadas.

Crear la funcion de preprocesamiento que incluye:

1. Tokenizacion
2. Eliminar puntuacion
3. Eliminar stopwords
4. Steamming
5. BOW + IF-IDF

Se utilizó una representación tipo Bag of Words (BOW), en la cual cada documento se representa mediante un conjunto de términos, asignando a cada término un peso calculado mediante TF-IDF.

In [ ]:
def text_preprocess(text):
    tokenizer = RegexpTokenizer(r'\w+')
    stemmer = PorterStemmer()
    
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in stopwords.words('spanish')]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

In [ ]:
preprocess = FunctionTransformer(
    lambda x: x.apply(text_preprocess)
)

In [ ]:
pipeline = Pipeline([
    ('preprocess', preprocess),
    ('tfidf', TfidfVectorizer())
])

Tiene sentido hacerlo primero sobre X train para evaluar el modelo completo desde la seleccion del vocabulario. 

In [ ]:
X_tfidf = pipeline.fit_transform(X_train)

2. A partir de la matriz TF-IDF construida en el paso anterior, aplica el algoritmo SVD truncado (TruncatedSVD de scikit-learn) para obtener un modelo de tópicos mediante análisis semántico latente (LSA). Explora un número reducido de componentes (por ejemplo, entre 10 y 20) y, para al menos 5 de ellas, identifica y muestra las palabras con mayor peso (loadings), a modo de "tópicos". Interpreta cualitativamente si estos tópicos guardan relación con algunos de los 17 ODS trabajados en el proyecto.

In [ ]:
for n in [10, 15, 20]:
    svd = TruncatedSVD(n_components=n, random_state=32)
    svd.fit(X_tfidf)
    print(f"{n} componentes -> varianza explicada: {svd.explained_variance_ratio_.sum():.4f}")

In [ ]:
tsvd = TruncatedSVD(n_components=15, random_state=32)
X_lsa = tsvd.fit_transform(X_tfidf)

In [ ]:
vocab = prep_pipeline.named_steps['tfidf'].get_feature_names_out()
n_top = 10

for i, comp in enumerate(tsvd.components_):
    top_pos = np.argsort(comp)[::-1][:n_top]   # pesos más altos
    top_neg = np.argsort(comp)[:n_top]         # pesos más negativos
    print(f"\nTópico {i}")
    print("  (+):", ", ".join(vocab[top_pos]))
    print("  (-):", ", ".join(vocab[top_neg]))